In [1]:
import torch
from torch.nn import functional as F
from kerops.ops.linear.relu_linear_add import ReLULinearAdd, autotune_relu_lin_add, generate_inputs_relu_lin_add
from kerops.ops.assets import ASSETS_ROOT

In [2]:
autotune_relu_lin_add(ASSETS_ROOT / 'ReLULinAdd.toml', n_jobs_precompile=4)

Problem sizes:   0%|          | 0/4 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/54 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/54 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/54 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/54 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/54 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/54 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/54 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/54 [00:00<?, ?it/s]

In [59]:
channels = 128

x, weight, add_other = generate_inputs_relu_lin_add({'in_channels': channels})

In [60]:
%%timeit -r 10 -n 10
ReLULinearAdd(x, weight, add_other)
torch.cuda.synchronize()

258 μs ± 26.6 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [61]:
conv = torch.nn.Conv3d(x.shape[1], add_other.shape[1], kernel_size=1, bias=False, device='cuda', dtype=torch.float16)

In [62]:
%%timeit -r 10 -n 10
with torch.amp.autocast('cuda'), torch.inference_mode():
    o = F.relu(x)
    o = conv(o)
    o += add_other
torch.cuda.synchronize()

549 μs ± 34 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)
